# Clase 9 - ¿Cómo aprende un modelo a continuar un texto?
Procesamiento de Lenguaje Natural - Felipe Castillo Rendón - Campus Monterrey

Vamos a partir de «no puedo» y seguir el recorrido hasta una continuación. Veremos qué recibe el modelo, qué intenta predecir y cómo interpretar sus errores. Las demostraciones D1–D8 están resueltas; E1 es el trabajo del equipo.

## Archivos que acompañan al notebook

Vamos a trabajar con `solicitudes.csv`, que contiene los mensajes, y `apoyo_pln.py`, que contiene las funciones de apoyo. Ambos archivos van al mismo nivel que el notebook; no se necesitan carpetas ni un ZIP de recursos.

En Colab, abre el notebook y ejecuta la siguiente celda. Selecciona ambos archivos cuando aparezca el botón de carga. También puedes subirlos desde el panel Archivos, directamente a la carpeta inicial de la sesión. Si ya están cargados, la celda los reutiliza.


In [1]:
#@title Cargar los archivos de la práctica
from pathlib import Path
import sys

# El CSV y las funciones se buscan en la carpeta actual de la sesión.
ROOT = Path.cwd()
necesarios = ("solicitudes.csv", "apoyo_pln.py")
faltantes = [nombre for nombre in necesarios if not (ROOT / nombre).is_file()]
if faltantes:
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError("Coloca junto al notebook: " + ", ".join(faltantes)) from None
    print("Selecciona los archivos que faltan: " + ", ".join(faltantes))
    files.upload()
faltantes = [nombre for nombre in necesarios if not (ROOT / nombre).is_file()]
if faltantes:
    raise FileNotFoundError("Todavía falta cargar: " + ", ".join(faltantes))
# La ruta explícita permite importar el apoyo sin instalar un paquete del curso.
sys.path.insert(0, str(ROOT))
print("Listo: solicitudes.csv y apoyo_pln.py están en la carpeta de trabajo.")


Selecciona los archivos que faltan: solicitudes.csv, apoyo_pln.py


Saving apoyo_pln.py to apoyo_pln.py
Saving solicitudes.csv to solicitudes.csv
Listo: solicitudes.csv y apoyo_pln.py están en la carpeta de trabajo.


## Entorno de ejecución

Usaremos NumPy, pandas, scikit-learn, Matplotlib y PyTorch. En una instalación local con Python 3.11 puedes instalar las dependencias de `requirements_clases_07_10.txt` y seleccionar ese entorno como kernel. En Colab se utilizan las bibliotecas de la sesión; la siguiente celda muestra sus versiones.

La práctica utiliza CPU y no necesita una clave de acceso ni descargar un modelo. El modo `minimo` conserva el recorrido de clase. Una semilla fija el inicio del azar, aunque las versiones pueden producir pequeñas diferencias numéricas.


In [2]:
import sys, os, platform, time
from importlib.metadata import version
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from apoyo_pln import load_requests, seed_all, words
from apoyo_pln import vocabulary, TinyGRU
seed_all(42)
MODE = "minimo"  # CPU y 20 épocas para poder discutir las salidas en clase.
pd.set_option("display.max_colwidth", 100)
print("Python:", platform.python_version(), "| Dispositivo: CPU")
print({p: version(p) for p in ["numpy", "pandas", "scikit-learn", "torch"]})


Python: 3.13.18 | Dispositivo: CPU
{'numpy': '2.4.6', 'pandas': '2.2.3', 'scikit-learn': '1.6.1', 'torch': '2.11.0+cpu'}


## D1 - De una probabilidad a una pérdida

Empecemos con dos predicciones inventadas para poder revisar cada cuenta. El modelo asigna 0.5 al primer token correcto y 0.25 al segundo. Un **token** es una unidad de texto; aquí puede ser una palabra o un signo.

`ln` es el logaritmo natural: responde a qué potencia hay que elevar e≈2.71828 para obtener el número. Como e⁻⁰·⁶⁹³¹⁴⁷≈0.5, ln(0.5)≈−0.693147. En Python usamos `np.log`; no `log10`.

**Entrada:** las probabilidades [0.5, 0.25]. **Qué hacemos:** calculamos −ln de cada una, sumamos y dividimos entre dos. La pérdida penaliza más una probabilidad pequeña para la respuesta correcta. La **perplejidad** es exp(pérdida media).

**Antes de ejecutar:** ¿cuál de las dos predicciones recibirá la mayor penalización?

> La penalización más grande la recibe 0.25, dado que cuanto más pequeña es la probabilidad asignada al token correcto, más grande es −ln(p).




In [3]:
probabilidades = np.array([0.5, 0.25])
perdidas = -np.log(probabilidades)
H = perdidas.mean()
ppl = np.exp(H)
display(pd.DataFrame({"p_correcta": probabilidades, "ln(p)": np.log(probabilidades), "perdida": perdidas}))
print(f"P(secuencia) = 0.5 × 0.25 = {probabilidades.prod():.6f}")
print(f"H = ({perdidas[0]:.6f} + {perdidas[1]:.6f}) / 2 = {H:.6f}")
print(f"Perplejidad = exp({H:.6f}) = {ppl:.6f}")
assert np.isclose(ppl, np.sqrt(8))


,p_correcta,ln(p),perdida
0,0.50,-0.693147,0.693147
1,0.25,-1.386294,1.386294


P(secuencia) = 0.5 × 0.25 = 0.125000
H = (0.693147 + 1.386294) / 2 = 1.039721
Perplejidad = exp(1.039721) = 2.828427


### Leamos el resultado

La tabla tiene una fila por predicción. −ln(0.25)=1.386294, el doble de −ln(0.5)=0.693147. La media es 1.039721 y exp(1.039721)=2.828427. Conservamos todos los decimales en el cálculo y redondeamos al mostrar.

Si cuatro opciones fueran igualmente probables, p=1/4 para la correcta y exp(−ln(1/4))=4. En el ejemplo no hay literalmente 2.83 palabras: es un resumen de incertidumbre, no un porcentaje de aciertos.

**Después:** ¿por qué una perplejidad menor no garantiza una continuación verdadera?

> Supongo que es porque la perplejidad solo mide qué tan probable le resulta al modelo la continuación según lo aprendido, no si esa continuación es verdadera o útil. Al tratarse de una medida, cálculos sobre probabilidades, un modelo puede tener perplejidad baja y aun así generar algo falso, repetitivo o incoherente si eso es lo que refleja el corpus.

## D2 - La entrada y su objetivo están desplazados

Vamos a construir la tarea antes de entrenar. En este ejemplo didáctico usamos IDs, números que identifican tokens: 2=«no», 5=«puedo», 7=«pagar» y 9=«hoy». No son los IDs del corpus real.

**Entrada:** [2,5,7,9]. `[:-1]` toma todo salvo el último elemento; `[1:]` toma todo salvo el primero. **Salida:** tres parejas entrada→siguiente token.

**Antes:** si entrada y objetivo fueran iguales, ¿qué tarea aprendería la red?

> Aprendería a ser como un loro: repetir lo que recibe, es decir, copiar el input como output en cada posición en vez de predecir el siguiente token.

In [4]:
secuencia = torch.tensor([2, 5, 7, 9])
entrada = secuencia[:-1]
objetivo = secuencia[1:]
display(pd.DataFrame({"posición": [1,2,3], "entrada_ID": entrada.numpy(), "objetivo_ID": objetivo.numpy()}))
assert entrada.tolist() == [2,5,7] and objetivo.tolist() == [5,7,9]


,posición,entrada_ID,objetivo_ID
0,1,2,5
1,2,5,7
2,3,7,9


### Leamos el resultado

Leemos cada fila como una pregunta: después de 2 queremos 5; después de [2,5] queremos 7; después de [2,5,7] queremos 9. El modelo recurrente conserva un estado con información previa. No recibe el token futuro para predecirlo.

Copiar entrada como objetivo cambiaría la tarea: enseñaría a reproducir el token actual. Una pérdida baja en esa tarea no demostraría que aprendió a continuar texto.

## D3 - Actualizar un estado: primero una cuenta pequeña

Una **RNN** (red neuronal recurrente) actualiza un estado al leer cada token y reutiliza los mismos pesos. Aquí reducimos todo a un número para ver la operación.

**Entrada:** x=1; estado previo=0.5; pesos 0.2 y 0.4. Calculamos 0.2×1+0.4×0.5=0.4. Después aplicamos **tanh**, tangente hiperbólica: tanh(a)=(eᵃ−e⁻ᵃ)/(eᵃ+e⁻ᵃ), que limita la salida entre −1 y 1.

También revisamos una memoria LSTM con compuertas ya dadas. Una **compuerta** regula cuánto conservar; no es una decisión escrita a mano en el modelo entrenado.

**Antes:** ¿la memoria nueva 0.9 y el estado de salida tendrán que ser iguales?

> No, no son iguales: la memoria nueva es 0.9, pero el estado de salida aplica una compuerta y tanh sobre ella, por lo que el estado entregado difiere.

In [5]:
suma = 0.2 * 1 + 0.4 * 0.5
estado_rnn = np.tanh(suma)
memoria_lstm = 0.8 * 1 + 0.2 * 0.5
estado_lstm = 0.9 * np.tanh(memoria_lstm)
print(f"RNN: tanh({suma:.1f}) = {estado_rnn:.6f}")
print(f"LSTM: memoria = {memoria_lstm:.6f}; estado = {estado_lstm:.6f}")
assert np.isclose(estado_rnn, 0.3799489623)
assert np.isclose(estado_lstm, 0.644668083)


RNN: tanh(0.4) = 0.379949
LSTM: memoria = 0.900000; estado = 0.644668


### Leamos el resultado

En la RNN el estado nuevo es 0.379949. En la LSTM, 0.8×1+0.2×0.5=0.9 es la memoria; el estado entregado es 0.9×tanh(0.9)=0.644668. Estos valores enseñan el mecanismo; no proceden de un entrenamiento.

Entrenaremos una **GRU**, unidad recurrente con compuertas. Usa un estado y compuertas de actualización y reinicio; no reproduce exactamente la cuenta de memoria LSTM. No necesitamos entrenar las tres arquitecturas para entender esta comparación.

## D4 - Del corpus a bloques de ocho predicciones

**Entrada:** `solicitudes.csv`. `id` identifica la solicitud; `text` contiene el mensaje; `label` indica su categoría; `family` reúne variantes; `split` separa train, validation y test. Hoy el objetivo sale del propio texto, por eso no entrenamos con `label`.

Aprendemos el vocabulario solo con **train**, los datos de entrenamiento. Reservamos ID 0 para PAD (relleno) e ID 1 para UNK (token desconocido). `words` convierte a minúsculas y separa palabras y signos; conservamos su criterio.

Concatenamos los textos y tomamos nueve IDs: ocho serán entrada y los ocho siguientes serán objetivo. Avanzamos ocho posiciones; los bloques comparten el token que une ambos extremos.

**Antes:** ¿por qué añadir al vocabulario palabras de validación cambiaría el experimento?

> Porque se contaminaría el experimento. El vocabulario debe aprenderse solo con train para que validation mida generalización sobre tokens no vistos. Si añadiéramos palabras de validación, el modelo tendría información del conjunto de evaluación desde la construcción del vocabulario, y ya no podríamos atribuir el desempeño a que generaliza correctamente.



In [6]:
df = load_requests()
train = df[df.split == "train"]
validation = df[df.split == "validation"]
seed_all(42)
vocab = vocabulary(train.text)
inverse = {i: token for token, i in vocab.items()}
train_tokens = [vocab[w] for texto in train.text for w in words(texto)] + [1]
# Conservamos el corte original de la práctica para mantener el experimento.
blocks = [train_tokens[i:i+9] for i in range(0, len(train_tokens)-9, 8)]
batch = torch.tensor(blocks)
x, y = batch[:, :-1], batch[:, 1:]
print("Mensajes por partición:", df.split.value_counts().to_dict())
print("Vocabulario:", len(vocab), "| Bloques:", len(batch), "| Predicciones:", y.numel())
display(train[["id", "text"]].head(3))
display(pd.DataFrame({"posición": range(1,9), "entrada": [inverse[int(i)] for i in x[0]],
                      "objetivo": [inverse[int(i)] for i in y[0]]}))
assert torch.equal(x[:, 1:], y[:, :-1])


Mensajes por partición: {'train': 96, 'validation': 24, 'test': 24}
Vocabulario: 151 | Bloques: 166 | Predicciones: 1328


,id,text
0,S001,Olvidé la contraseña y necesito restablecerla.
1,S002,"Buen día, olvidé la contraseña y necesito restablecerla. ¿Me pueden orientar?"
2,S003,"Olvidé la contraseña y necesito restablecerla; ya reinicié y sigue igual, urge tantito."


,posición,entrada,objetivo
0,1,olvidé,la
1,2,la,contraseña
2,3,contraseña,y
3,4,y,necesito
4,5,necesito,restablecerla
5,6,restablecerla,.
6,7,.,buen
7,8,buen,día


### Leamos el resultado

La tabla permite leer la primera secuencia sin descifrar IDs. Los nombres de las particiones conservan su uso: train ajusta pesos, validation permite observar desempeño fuera de train y test queda reservado.

Hay una simplificación importante: no insertamos límites de mensaje. Una pareja puede unir el final de una solicitud con el inicio de otra. Además, el corte original puede dejar un tramo final sin utilizar; el UNK añadido al final no es un token de fin de frase. No lo confundas con EOS (fin de secuencia).

## D5 - Seguir las formas de los datos dentro de la GRU

**Entrada:** dos bloques de ocho IDs. Un **embedding** convierte cada ID en un vector de 16 números aprendidos. La GRU entrega un estado de 32 números por posición. La capa lineal convierte cada estado en V **logits**, puntuaciones todavía sin normalizar, una por token del vocabulario.

B significa número de bloques; L, posiciones; V, tamaño del vocabulario. Así, B×L×V se lee «bloques, posiciones, opciones».

**Antes:** si duplicamos B, ¿debería cambiar V?

> No, B se puede duplicar, pero V se mantiene porque es el tamaño del vocabulario y queda fijo. V solo cambiaría si cambiáramos el vocabulario.

In [7]:
model = TinyGRU(len(vocab))
embedded = model.embedding(x[:2])
hidden, last_state = model.rnn(embedded)
logits = model.head(hidden)
print("IDs:", tuple(x[:2].shape))
print("Embeddings:", tuple(embedded.shape))
print("Estados por posición:", tuple(hidden.shape), "| Estado final:", tuple(last_state.shape))
print("Logits:", tuple(logits.shape))
assert torch.allclose(model(x[:2]), logits)
opt = torch.optim.Adam(model.parameters(), lr=0.025)
history = []
epochs = 20 if MODE == "minimo" else 70


IDs: (2, 8)
Embeddings: (2, 8, 16)
Estados por posición: (2, 8, 32) | Estado final: (1, 2, 32)
Logits: (2, 8, 151)


### Leamos el resultado

Con B=2 y L=8, los IDs tienen forma (2,8), los embeddings (2,8,16) y los estados (2,8,32). La salida contiene V puntuaciones en cada una de las 16 posiciones.

`last_state` guarda el último estado de cada secuencia; `hidden` contiene todos los pasos. Usamos `hidden` para poder calcular una pérdida por posición. El estado comienza de nuevo en cada bloque porque no pasamos uno previo a la GRU.

## D6 - Entrenar: predecir, medir el error y actualizar

**Entrada:** `x`, `y`, el modelo y el optimizador Adam, que ajusta los pesos. Una **época** recorre los bloques de entrenamiento; aquí todos caben en un lote.

`reshape` reúne bloques y posiciones: B×8×V se convierte en (B×8)×V. No mezcla las etiquetas: `y` se aplana en el mismo orden. Entropía cruzada recibe logits, por eso no aplicamos softmax antes de pasarle los datos.

En cada época borramos gradientes anteriores, calculamos pérdida, propagamos el error hacia los pesos y limitamos la norma del gradiente a 1. Ese recorte evita pasos excesivos; no limita la pérdida a 1. Finalmente `step` actualiza los pesos.

**Antes:** ¿una pérdida menor en train demuestra que el modelo generaliza?

> No necesariamente, una pérdida menor en train indica mejor ajuste a los datos de entrenamiento, lo que puede incluir memorización, pero no garantiza por sí solo que generalice. Al compararla con validation loss se puede determinar que generaliza si ambas son bajas y el gap entre ellas es pequeño.


In [8]:
start = time.perf_counter()
for epoch in range(epochs):
    model.train()
    opt.zero_grad()
    z = model(x)
    loss = torch.nn.functional.cross_entropy(z.reshape(-1, len(vocab)), y.reshape(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    history.append(loss.item())
print("Épocas:", epochs, "| Segundos:", round(time.perf_counter() - start, 3))
display(pd.DataFrame({"época": range(1,epochs+1), "pérdida_antes_de_actualizar": history}).iloc[[0,4,9,-1]])


Épocas: 20 | Segundos: 0.195


,época,pérdida_antes_de_actualizar
0,1,4.987799
4,5,3.812286
9,10,2.500947
19,20,1.019159


### Leamos el resultado

Cada fila muestra la pérdida calculada antes de actualizar los pesos en esa época. Por eso la última fila no es exactamente la evaluación del modelo después de la última actualización. La calcularemos enseguida.

La caída de pérdida muestra mejor ajuste al objetivo de entrenamiento. Puede incluir memorización y transiciones artificiales. El tiempo depende del equipo: no es una medida de calidad lingüística.

## D7 - Evaluar con el mismo tamaño de contexto

**Entrada:** el modelo ya entrenado, sus bloques de train y los textos de validation. Mantenemos el vocabulario congelado: un token nuevo se representa con UNK=1.

Para comparar, construimos bloques de ocho predicciones también en validación y reiniciamos el estado entre bloques. `eval()` selecciona el modo de evaluación y `no_grad()` evita calcular gradientes; ninguno modifica los pesos.

**Salida:** una fila por partición con predicciones evaluadas, entropía cruzada media (CE) y perplejidad (PPL). Indicamos cuántos objetivos son UNK y cuántas transiciones quedan fuera del corte. No evaluamos test.

**Antes:** ¿sería justo comparar ocho pasos con un texto completo que conserva todo su contexto?

> No sería justo, dado que en este tipo de modelos el contexto es clave para los cálculos y las predicciones que se derivan de él. Comparar ocho pasos con un texto completo que conserva todo su contexto cambiaría las condiciones, pues no es justo usar métricas que no contemplan lo mismo que vio el modelo en cada caso.


In [9]:
validation_tokens = [vocab.get(w, 1) for texto in validation.text for w in words(texto)]
val_blocks = [validation_tokens[i:i+9] for i in range(0, len(validation_tokens)-9, 8)]
val_batch = torch.tensor(val_blocks)
vx, vy = val_batch[:, :-1], val_batch[:, 1:]
model.eval()
with torch.no_grad():
    ce_train = torch.nn.functional.cross_entropy(model(x).reshape(-1,len(vocab)), y.reshape(-1)).item()
    ce_val = torch.nn.functional.cross_entropy(model(vx).reshape(-1,len(vocab)), vy.reshape(-1)).item()
metricas = pd.DataFrame([
    {"partición":"train", "objetivos":y.numel(), "CE":ce_train, "PPL":np.exp(ce_train)},
    {"partición":"validation", "objetivos":vy.numel(), "CE":ce_val, "PPL":np.exp(ce_val)}])
display(metricas)
unknown = int((vy == 1).sum())
print(f"Objetivos desconocidos en validación: {unknown}/{vy.numel()} = {unknown/vy.numel():.2%}")
print("Transiciones sin evaluar por el corte:", len(validation_tokens)-1-vy.numel())
# Conservamos la medición anterior como contraste: aquí el estado recorre todo el texto.
with torch.no_grad():
    ce_cadena = torch.nn.functional.cross_entropy(model(torch.tensor([validation_tokens[:-1]])).squeeze(0),
                                                  torch.tensor(validation_tokens[1:])).item()
print(f"Referencia con cadena completa: CE={ce_cadena:.6f}; PPL={np.exp(ce_cadena):.6f}")


,partición,objetivos,CE,PPL
0,train,1328,0.933755,2.544044
1,validation,336,3.670425,39.268589


Objetivos desconocidos en validación: 84/336 = 25.00%
Transiciones sin evaluar por el corte: 6
Referencia con cadena completa: CE=3.616691; PPL=37.214235


### Leamos el resultado

Para comparar train y validation usa las dos filas de la tabla, que comparten tamaño de bloque y reinicio de estado. La referencia de cadena completa usa otro contexto y otro conjunto de posiciones: no atribuyas toda la diferencia a la calidad del modelo.

Una tasa alta de UNK limita la interpretación: muchas palabras distintas se agrupan en un solo ID. La perplejidad tampoco equivale a porcentaje de aciertos. Antes de comparar con otro experimento, revisa texto, tokenización, vocabulario, bloques y denominador.

## D8 - Generar: elegir una continuación y repetir

**Entrada:** el prefijo «no puedo», que significa el texto inicial disponible. En cada paso calculamos los logits de la última posición y elegimos su mayor valor con `argmax`. Esta elección se llama **greedy**, o voraz.

Añadimos el ID elegido al prefijo y repetimos ocho veces. No entrenamos de nuevo ni utilizamos las etiquetas de soporte. El código vuelve a procesar el prefijo completo en cada paso; es sencillo para aprender, aunque no es la implementación más eficiente.

**Antes:** ¿elegir siempre la opción más probable evitará las repeticiones?

> No, si utilizamos un algoritmo greedy no garantizamos que la opción obtenida sea la más eficiente (en este caso, a nivel secuencia, la que evite menos repeticiones). Más bien, nos garantiza obtener la mejor opción en el momento (no "piensa" a largo plazo, se enfoca en la predicción siguiente).



In [10]:
prefix = "no puedo"
ids = [vocab.get(w, 1) for w in words(prefix)]
pasos = []
for paso in range(1,9):
    with torch.no_grad():
        puntuaciones = model(torch.tensor([ids]))[0,-1]
        next_id = int(puntuaciones.argmax())
    ids.append(next_id)
    pasos.append({"paso":paso, "token_elegido":inverse[next_id]})
display(pd.DataFrame(pasos))
generacion = " ".join(inverse[i] for i in ids)
print("Texto completo:", generacion)


,paso,token_elegido
0,1,conectarme
1,2,a
2,3,mi
3,4,método
4,5,de
5,6,pago
6,7,.
7,8,buen


Texto completo: no puedo conectarme a mi método de pago . buen


### Leamos el resultado

La tabla tiene una fila por token añadido. La secuencia completa conserva el prefijo y agrega ocho tokens. Una continuación probable puede repetir plantillas o perder sentido; no comprueba que sea una respuesta útil ni verdadera.

**Después:** señala una transición concreta que se pueda explicar por el corpus. Separa lo observado de tu hipótesis sobre su causa.

> Con el texto `no puedo conectarme a mi método de pago . buen` es posible ver que el output obtenido no es siempre una continuación coherente. Después del "." aparece lo que parece ser el inicio de otro mensaje del corpus. Eso sugiere una transición artificial, probablemente porque el corpus concatena mensajes sin límites.

## E1 - ¿Qué cambia al cambiar el prefijo?
Mantengan el modelo entrenado y comparen las tres continuaciones más probables de al menos dos prefijos elegidos por el equipo.

1. Antes de ejecutar, escriban qué esperan que cambie y por qué.
2. Conviertan cada prefijo a IDs usando el mismo vocabulario. Revisen si aparece UNK.
3. Apliquen `softmax` a los logits de la última posición para obtener probabilidades y `topk(3)` para obtener tres opciones.
4. Organicen prefijo, continuación y probabilidad en una tabla. No vuelvan a entrenar.
5. Expliquen una diferencia concreta y una limitación. La suma de tres opciones no tiene que ser 1: faltan las demás palabras del vocabulario.

Esta sección es trabajo del equipo; las demostraciones anteriores sirven como apoyo.

In [13]:
# E1 - Escriban aquí su comparación. Mantengan el mismo modelo y vocabulario.
# Dejen visibles la tabla y su interpretación.

model.eval()
prefijos = ["no puedo", "necesito cambiar mi"]

filas = []
for prefijo in prefijos:
    ids_prefijo = [vocab.get(w, 1) for w in words(prefijo)]
    unk = [w for w in words(prefijo) if vocab.get(w, 1) == 1]
    with torch.no_grad():
        logits_ult = model(torch.tensor([ids_prefijo]))[0, -1]
    probs = torch.softmax(logits_ult, dim=-1)
    top_probs, top_ids = probs.topk(3)
    for p, i in zip(top_probs.tolist(), top_ids.tolist()):
        filas.append({
            "prefijo": prefijo,
            "continuación": inverse[i],
            "probabilidad": round(p, 4),
            "UNK_en_prefijo": ", ".join(unk) if unk else "ninguno",
        })

tabla_e1 = pd.DataFrame(filas)
display(tabla_e1)

,prefijo,continuación,probabilidad,UNK_en_prefijo
0,no puedo,conectarme,0.1685,ninguno
1,no puedo,instalar,0.1248,ninguno
2,no puedo,entrar,0.1216,ninguno
3,necesito cambiar mi,método,0.2586,ninguno
4,necesito cambiar mi,de,0.1647,ninguno
5,necesito cambiar mi,correo,0.0823,ninguno


### Nuestra explicación
- **Predicción inicial**: La suposición que tengo es que el prefijo "no puedo" tienda a continuar con verbos o frases relacionados con la incapacidad de hacer u obtener algo (conectarme, pagar, entrar), mientras que "necesito cambiar mi" debería inclinarse hacia temas de soporte (cuenta, pago, contraseña), porque el contexto cambia la última posición y por tanto los logits.

- **Prefijos y tokens desconocidos**: Ambos usan palabras presentes en el vocabulario de train, etonces no se presentan.

- **Una diferencia respaldada por cifras**: Una diferencia respaldada por cifras: para "no puedo" el top-3 se reparte entre conectarme (0.1685), instalar (0.1248) y entrar (0.1216), una masa más dispersa; para "necesito cambiar mi" se concentra en método (0.2586), de (0.1647) y correo (0.0823), con la opción más probable claramente por encima. En ningún prefijo las tres probabilidades suman 1, porque faltan las demás palabras del vocabulario.

- **Lo que podemos concluir**: El modelo es sensible al prefijo debido a que, para este tipo de modelos/redes, el contexto es clave para el cálculo de predicciones.

- **Lo que todavía no podemos afirmar**: No se puede afirmar aún que la continuación más probable sea verdadera, útil o gramaticalmente completa. El top 3 no es una validación de la secuencia, por lo que una probabilidad alta solo indica que el modelo "ya lo esperaba" dado al reconocimiento de patrones, no que el texto sea necesariamente correcto.

## Cierre - explica una cuenta y una decisión
Una persona explica cómo obtuvo pérdida y perplejidad; otra sigue un token desde el ID hasta la salida. El equipo distingue ajuste a train, validación y una muestra generada. Guarda el notebook con tus salidas y respuestas antes de cerrar Colab.

Un modelo secuencia a secuencia (seq2seq) transforma una entrada en otra secuencia. Un codificador procesa la entrada y un decodificador genera la salida. La atención permite consultar distintas posiciones: será el siguiente paso del curso. Hoy entrenamos un predictor de siguiente token con GRU.

> La pérdida y perplejidad se obtuvo al calcular −ln(p) para cada probabilidad del token correcto (p=0.25 da 1.386294 frente a p=0.5 con 0.693147), se promedió la pérdida media (1.039721), y la perplejidad es exp de esa pérdida (2.828427). El proceso que se sigue es el siguiente: un token entra como ID, el embedding lo convierte en vector de 16, la GRU produce un estado de 32 por posición, la capa lineal lo transforma en V=151 logits, y con softmax/argmax se elige el siguiente token, que se añade al prefijo y se repite. Para trabajar con este tipo de modelos, es importante distinguir entre las funciones de cada parte que lo conforman: train (CE=0.933755, PPL=2.544043) es la sección que ajusta conforme a los datos a los que se le expone, con posible memorización; validation (CE=3.670425, PPL=39.268589) mide generalización con vocabulario congelado y 25% de UNK, lo que limita su interpretación, mientras que la muestra generada ("no puedo conectarme a mi método de pago . buen") es una salida concreta y no debe leerse como una métrica.

Referencias: documentación oficial de PyTorch, `torch.nn.GRU` y `torch.nn.CrossEntropyLoss`; Jurafsky y Martin, *Speech and Language Processing*, capítulos de modelado del lenguaje y redes recurrentes. Las funciones y versiones utilizadas quedan registradas arriba.

Documentación oficial: [GRU](https://pytorch.org/docs/stable/generated/torch.nn.GRU.html), [entropía cruzada](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) y [Speech and Language Processing](https://web.stanford.edu/~jurafsky/slp3/).